# Bước 00: Hotfix Căn Chỉnh Thời Tiết Causal (Re-align Weather Hotfix)
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

INPUT_PATH = Path("../../data/mlmart_base/v3_preprocessing.parquet")
OUTPUT_PATH = Path("../../data/mlmart_base/v3_preprocessing_hotfix.parquet")

WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
print("Đã nạp thư viện thành công.")
print(f"Đường dẫn INPUT: {INPUT_PATH}")

Đã nạp thư viện thành công.
Đường dẫn INPUT: ../../data/mlmart_base/v3_preprocessing.parquet


In [2]:
# ── 1. Đọc dữ liệu ──
df = pd.read_parquet(INPUT_PATH)
print(f"Đã đọc file đầu vào chưa hotfix: {INPUT_PATH}")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['weather_timestamp'] = pd.to_datetime(df['weather_timestamp'])

_delta_truoc = (df['weather_timestamp'] - df['timestamp']).dt.total_seconds() / 60
_leak_truoc = int((_delta_truoc > 0).sum())
print(f"Tổng số dòng: {len(df):,}")
print(f"Dòng dùng thời tiết TƯƠNG LAI trước khi sửa: {_leak_truoc:,}/{len(df):,} ({_leak_truoc / len(df) * 100:.2f}%)")

Đã đọc file đầu vào chưa hotfix: ../../data/mlmart_base/v3_preprocessing.parquet
Tổng số dòng: 2,731,946
Dòng dùng thời tiết TƯƠNG LAI trước khi sửa: 1,964,180/2,731,946 (71.90%)


## Hotfix 2: Khôi phục sản lượng buổi tối bị ETL ghi nhầm về 0

ETL thượng nguồn ép sản lượng về `0` từ **18:30** trở đi. Đối chiếu với tệp thô
`data/raw/Solar_Energy_Generation.csv` cho thấy đây là **số đo thật bị mất**, không phải
trạm ngừng phát:

| Giờ | Tệp thô | ML Mart |
|---|---|---|
| 18:15 | 5,956 | 5,855 |
| **18:30** | **4,844** | **0,000** |
| 19:00 | 3,095 | 0,000 |
| 20:30 | 0,326 | 0,000 |

Đường tắt nắng trong tệp thô giảm dần mượt mà, đúng vật lý. Bước dưới đây lấy lại đúng
giá trị đã đo, **không nội suy, không ước lượng**, và chỉ ghi đè những dòng thoả đồng thời
ba điều kiện: từ 18:30 trở đi, mart đang ghi 0, và tệp thô có giá trị dương.


In [3]:
# ── 2. Khôi phục sản lượng buổi tối bị ETL ghi nhầm về 0 ──
# Chỉ lấy lại số ĐO THẬT từ tệp thô, không nội suy. Có cổng kiểm tra ở cuối ô:
# tuyệt đối không được đổi bất kỳ dòng nào trước 18:30.
GIO_CAT = 18.5
RAW_PATH = '../../data/raw/Solar_Energy_Generation.csv'

_tho = pd.read_csv(
    RAW_PATH,
    usecols=['SiteKey', 'Timestamp', 'SolarGeneration'],
    parse_dates=['Timestamp'],
).rename(columns={'SiteKey': 'site_id',
                  'Timestamp': 'timestamp',
                  'SolarGeneration': '_san_luong_tho'})
_tho = _tho.groupby(['site_id', 'timestamp'], as_index=False)._san_luong_tho.max()

_n0 = len(df)
df = df.merge(_tho, on=['site_id', 'timestamp'], how='left')
assert len(df) == _n0, f"Ghép làm đổi số dòng: {_n0:,} -> {len(df):,}"

# Lấy mốc so sánh SAU khi merge: merge cấp lại chỉ số mới, nếu lấy trước thì
# hai Series lệch hàng khi df gốc không dùng chỉ số 0..n-1.
_truoc = df['energy_generated_kwh'].copy()
_gio = df['timestamp'].dt.hour + df['timestamp'].dt.minute / 60

_can_va = (
    (_gio >= GIO_CAT)
    & (df['energy_generated_kwh'].fillna(0) == 0)
    & (df['_san_luong_tho'].notna())
    & (df['_san_luong_tho'] > 0)
)
df.loc[_can_va, 'energy_generated_kwh'] = df.loc[_can_va, '_san_luong_tho']
df['va_san_luong_toi'] = _can_va          # giữ cờ để mọi bước sau truy ngược được
df = df.drop(columns=['_san_luong_tho'])

print(f"Số dòng khôi phục : {int(_can_va.sum()):,}")
print(f"Năng lượng lấy lại: {float(df.loc[_can_va, 'energy_generated_kwh'].sum()):,.0f} kWh")
print(f"Số trạm ảnh hưởng : {df.loc[_can_va, 'site_id'].nunique()}")
print(f"Tổng trước / sau  : {_truoc.sum():,.0f} -> {df['energy_generated_kwh'].sum():,.0f} kWh")

_he = df[(df['timestamp'] >= '2021-12-01') & (df['timestamp'] < '2022-01-01')]
_gh = _he['timestamp'].dt.hour + _he['timestamp'].dt.minute / 60
print("\nTrung bình theo khung giờ chiều tối, tháng 12/2021 (sau khi vá):")
print(_he[(_gh >= 18) & (_gh <= 20)]
      .groupby(_gh[(_gh >= 18) & (_gh <= 20)])['energy_generated_kwh'].mean().round(3).to_string())

# ── Cổng kiểm soát ──
_ngoai = (df['energy_generated_kwh'].fillna(-1) != _truoc.fillna(-1)) & (_gio < GIO_CAT)
assert not _ngoai.any(), f"Có {int(_ngoai.sum()):,} dòng TRƯỚC {GIO_CAT}h bị đổi - không được phép"
print(f"\n[ĐẠT] Không dòng nào trước {GIO_CAT}h bị thay đổi.")


Số dòng khôi phục : 87,844
Năng lượng lấy lại: 167,959 kWh
Số trạm ảnh hưởng : 42
Tổng trước / sau  : 9,143,053 -> 9,311,012 kWh



Trung bình theo khung giờ chiều tối, tháng 12/2021 (sau khi vá):
timestamp
18.00    6.645
18.25    5.855
18.50    4.557
18.75    3.644
19.00    2.917
19.25    2.091
19.50    1.491
19.75    0.968
20.00    0.537

[ĐẠT] Không dòng nào trước 18.5h bị thay đổi.


In [4]:
# ── Căn chỉnh lại thời tiết Causal (Code NGUYÊN BẢN từ srcs/00_utils/04_realign_mlmart_weather.py) ──
WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
LOOKUP_KEY = ("site_id", "_weather_hour")

# 1. Trích xuất bảng tra thời tiết chuẩn tại minute 00 (Hàm load_hourly_lookup)
_frame_m0 = df[pd.to_datetime(df['timestamp']).dt.minute.eq(0)].copy()
_frame_m0['_weather_hour'] = pd.to_datetime(_frame_m0['timestamp'], errors='raise')
_lookup = _frame_m0[[*LOOKUP_KEY, *[c for c in WEATHER_COLUMNS if c in df.columns]]].sort_values([*LOOKUP_KEY, 'weather_id'], kind='stable')
_lookup = _lookup.drop_duplicates(list(LOOKUP_KEY), keep='first').set_index(list(LOOKUP_KEY))

# 2. Realign thời tiết theo mốc timestamp.dt.floor('h') (Hàm realign_batch)
_timestamp = pd.to_datetime(df['timestamp'], errors='raise')
_keys = pd.MultiIndex.from_arrays(
    [df['site_id'].to_numpy(), _timestamp.dt.floor('h').to_numpy()],
    names=LOOKUP_KEY
)
_aligned = _lookup.reindex(_keys).reset_index(drop=True)

for col in WEATHER_COLUMNS:
    if col in df.columns:
        df[col] = _aligned[col].to_numpy()

df['weather_timestamp'] = _timestamp.dt.floor('h')

del _frame_m0, _lookup, _keys, _aligned
gc.collect()

0

In [5]:
# ── 3. Kiểm tra kết quả SAU khi sửa và Cổng kiểm soát (Assertion) ──
_delta_sau = (df['weather_timestamp'] - df['timestamp']).dt.total_seconds() / 60
_leak_sau = int((_delta_sau > 0).sum())
print(f"Dòng dùng thời tiết TƯƠNG LAI sau khi sửa: {_leak_sau:,} (Phải bằng 0)")

assert _leak_sau == 0, "LỖI BẢO VỆ: Vẫn còn dòng rò rỉ thời tiết tương lai!"
print("CỔNG KIỂM TRÁ: ĐẠT — 100% Causal Non-leaking Weather Data!")

Dòng dùng thời tiết TƯƠNG LAI sau khi sửa: 0 (Phải bằng 0)
CỔNG KIỂM TRÁ: ĐẠT — 100% Causal Non-leaking Weather Data!


In [6]:
# ── 4. Ghi file output riêng, không ghi đè nguồn ──
df.to_parquet(OUTPUT_PATH, index=False)
print(f"HOÀN TẤT: Đã ghi file hotfix riêng tại: {OUTPUT_PATH}")

HOÀN TẤT: Đã ghi file hotfix riêng tại: ../../data/mlmart_base/v3_preprocessing_hotfix.parquet
